# 02: Feature Engineering & Domain Expertise

This notebook explains the 15 interpretable features designed with domain expertise for intimate partner violence risk assessment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.case_generator import CaseGenerator
from src.data.feature_extractor import FeatureExtractor

sns.set_style('whitegrid')

## The 15 Interpretable Features

1. **violence_incidents** - Count of documented violent incidents (normalized)
2. **max_violence_severity** - Severity rating of most serious incident (0-1)
3. **violence_escalation_rate** - Trend in violence severity over time
4. **threat_frequency** - How often threats are made (normalized)
5. **weapon_mention** - Whether weapons have been involved (0-1)
6. **threat_severity_max** - Most severe threat documented
7. **isolation_indicators** - Social/financial isolation factors
8. **restraining_order_breaches** - Violation of protection orders
9. **police_contact_frequency** - Police involvement rate
10. **child_welfare_contact** - Child protection agency involvement (binary)
11. **days_since_last_incident** - Recency of last event (inverse normalized)
12. **trend_worsening** - Is situation getting worse? (0-1)
13. **velocity_of_escalation** - Speed of escalation (0-1)
14. **multiple_agencies_involved** - Count of different agencies handling case
15. **breach_of_protection** - Violation of protection measures

In [ ]:
# Generate and extract features
gen = CaseGenerator(seed=42)
cases = gen.generate_dataset(n_cases=500)
extractor = FeatureExtractor()
features_list = [extractor.extract_features(c) for c in cases]

df = pd.DataFrame(features_list)
df['risk_category'] = [c['risk_category'] for c in cases]

print(f"Total features extracted: {len(df.columns) - 1}")
print(f"\nFeature ranges (should all be 0-1 except risk_label):")
print(df.drop('risk_label', axis=1).drop('risk_category', axis=1).describe().round(3))

## Feature Importance by Risk Category

In [ ]:
# Compare high-risk vs low-risk feature values
high_risk = df[df['risk_category'] == 'high'].drop(['risk_label', 'risk_category'], axis=1).mean()
low_risk = df[df['risk_category'] == 'low'].drop(['risk_label', 'risk_category'], axis=1).mean()

comparison = pd.DataFrame({
    'High-Risk Avg': high_risk,
    'Low-Risk Avg': low_risk,
    'Difference': high_risk - low_risk
}).sort_values('Difference', ascending=False)

print("Feature Comparison (High-Risk vs Low-Risk):")
print(comparison.round(3))

## Visualization: Feature Distributions

In [ ]:
# Plot top 6 discriminative features
top_features = comparison['Difference'].head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    high_vals = df[df['risk_category'] == 'high'][feature]
    low_vals = df[df['risk_category'] == 'low'][feature]
    
    ax.hist([low_vals, high_vals], bins=15, label=['Low-Risk', 'High-Risk'],
            color=['green', 'red'], alpha=0.6, edgecolor='black')
    ax.set_title(f'{feature}')
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()